# A new (fictional) Indian female voice → RVC v2 model — Colab T4

No real person's voice is used: VoxCPM2 (open source) invents a voice, makes ~10 minutes of speech in it, and RVC v2
(Applio, MIT) learns that voice. The RVC model then turns any speech (Gemini / Piper / browser, Hindi or English)
into this voice for the website avatar.

**VS Code:** Select Kernel → Colab → New Colab Server → GPU (T4). **colab.google:** Runtime → Change runtime type → T4 GPU.
Run the cells in order. If a cell fails, the cause is inside the red error message — copy that text.
Total ≈ 45–70 min (data ~10 min, RVC install ~10 min, training ~20–40 min).

In [ ]:
#@title 0) Setup check: GPU
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
print(run("nvidia-smi --query-gpu=name,memory.total --format=csv"))

: 

In [ ]:
#@title 1) VoxCPM in its own Python 3.10 environment (~5 min; safe to run again)
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba")
envs = run("micromamba env list")
if " vox " not in envs and "/vox\n" not in envs:
    run("micromamba create -y -q -n vox -c conda-forge python=3.10")
run("micromamba run -n vox pip install -q -U voxcpm soundfile numpy", tail=40)
print(run("micromamba run -n vox python -c \"import voxcpm, torch; print('voxcpm ok | torch', torch.__version__, '| cuda', torch.cuda.is_available())\""))

In [ ]:
#@title 2) Make 6 new voices — listen and pick the best young Indian female voice
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
# generator script: runs inside the "vox" environment; takes a list of jobs {text, out, seed?, ref?, ref_text?}
open("/content/vox_gen.py", "w").write("""
import inspect, json, sys, time
import numpy as np, soundfile as sf, torch, torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True
from voxcpm import VoxCPM
jobs = json.load(open(sys.argv[1], encoding="utf-8"))
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=True, optimize=False)
SR = model.tts_model.sample_rate
SIG = inspect.signature(model.generate).parameters
print("generate() parameters:", list(SIG), flush=True)
t0, total = time.time(), 0.0
for i, j in enumerate(jobs):
    if j.get("seed") is not None:
        torch.manual_seed(j["seed"]); np.random.seed(j["seed"])
    kw = {"text": j["text"]}
    if "inference_timesteps" in SIG: kw["inference_timesteps"] = 10
    if j.get("ref"):
        k = next((k for k in ("prompt_wav_path", "reference_wav_path", "prompt_audio_path") if k in SIG), None)
        if not k: raise SystemExit("no reference-audio parameter in %s" % list(SIG))
        kw[k] = j["ref"]
        kt = next((k for k in ("prompt_text", "reference_text") if k in SIG), None)
        if kt: kw[kt] = j.get("ref_text", "")
    out = model.generate(**kw)
    parts = [out] if isinstance(out, (np.ndarray, torch.Tensor)) else list(out)
    w = np.concatenate([np.asarray(p.detach().cpu() if hasattr(p, "detach") else p, dtype=np.float32).reshape(-1) for p in parts])
    sf.write(j["out"], w, SR)
    total += len(w) / SR
    if i % 10 == 0 or i == len(jobs) - 1:
        print(f"{i + 1}/{len(jobs)}  {total / 60:.1f} min of speech  ({time.time() - t0:.0f} s)", flush=True)
""")
os.makedirs("/content/cand", exist_ok=True)
SEEDS = range(6)   # none good? use range(6, 12)
vox([{"text": GREETING, "out": f"/content/cand/voice_{s}.wav", "seed": s} for s in SEEDS])
for s in SEEDS:
    print(f"--- voice {s}"); ipd.display(ipd.Audio(f"/content/cand/voice_{s}.wav"))

In [ ]:
#@title 3) ~10 minutes of speech in the chosen voice (English, Hinglish, Hindi — showroom talk)
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
CHOSEN = 0   #@param {type:"integer"}
REF, REF_TEXT = f"/content/cand/voice_{CHOSEN}.wav", GREETING
need(REF, 2)
open("/content/chosen.txt", "w").write(str(CHOSEN))
models = ["Fortuner", "Innova Hycross", "Innova Crysta", "Glanza", "Urban Cruiser Hyryder", "Camry", "Legender", "Hilux",
          "Rumion", "Taisor", "Creta", "Venue", "Verna", "Alcazar", "Exter", "i20"]
en = ["The {m} is one of our most popular cars this month.", "Would you like to book a test drive for the {m}?",
      "The {m} comes with six airbags and a five star safety rating.", "I can show you the {m} page on our website right now.",
      "Our showroom in Moti Nagar is open from ten in the morning till seven in the evening.",
      "The on-road price depends on the variant and the insurance you choose.", "Can I have your name and phone number, please?",
      "Your test drive is confirmed for tomorrow at eleven thirty.", "Thank you for calling, have a wonderful day!",
      "The waiting period for the {m} is about four to six weeks right now.", "We also have easy EMI options starting from a low down payment.",
      "Is there anything else I can help you with today?"]
hinglish = ["{m} ke baare mein main aapko poori jaankari de sakti hoon.", "Kya aap {m} ki test drive book karna chahenge?",
            "Aap kal subah das baje showroom aa sakte hain.", "{m} mein petrol aur hybrid dono options milte hain.",
            "Aapka naam aur mobile number bata dijiye, please.", "Bilkul, main abhi aapke liye slot check karti hoon.",
            "Is mahine {m} par achha offer chal raha hai.", "Koi aur sawaal ho toh zaroor poochiye."]
hindi = ["नमस्ते! गैलेक्सी टोयोटा में आपका स्वागत है।", "आपकी टेस्ट ड्राइव कल सुबह ग्यारह बजे के लिए पक्की हो गई है।",
         "हमारा शोरूम मोती नगर में है और रविवार को भी खुला रहता है।", "क्या मैं आपकी और कोई मदद कर सकती हूँ?",
         "इस गाड़ी में सात एयरबैग और बड़ी डिक्की मिलती है।", "धन्यवाद! आपका दिन शुभ हो।"]
lines = []
for i, m in enumerate(models):
    lines += [en[i % len(en)].format(m=m), en[(i + 5) % len(en)].format(m=m), hinglish[i % len(hinglish)].format(m=m),
              hinglish[(i + 3) % len(hinglish)].format(m=m), hindi[i % len(hindi)]]
lines += [e.format(m="car") for e in en] + [h.format(m="ye gaadi") for h in hinglish]
os.makedirs("/content/dataset", exist_ok=True)
jobs = [{"text": t, "out": f"/content/dataset/{i:03d}.wav", "ref": REF, "ref_text": REF_TEXT} for i, t in enumerate(lines)]
# a test sentence in a DIFFERENT voice: RVC must turn it into the chosen voice in cell 6
jobs.append({"text": "Namaste! Main Galaxy Toyota se baat kar rahi hoon. Fortuner ki test drive kab rakhein?",
             "out": "/content/test_other_voice.wav", "seed": CHOSEN + 50})
vox(jobs)
os.remove("/content/dataset/" + f"{len(lines):03d}.wav") if os.path.exists("/content/dataset/" + f"{len(lines):03d}.wav") else None
print(len(glob.glob("/content/dataset/*.wav")), "clips in /content/dataset")
ipd.display(ipd.Audio("/content/dataset/002.wav"))

In [ ]:
#@title 4) RVC v2 (Applio) in its own Python 3.10 environment (~10 min)
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba")
if " rvc " not in run("micromamba env list") and "/rvc\n" not in run("micromamba env list"):
    run("micromamba create -y -q -n rvc -c conda-forge python=3.10")
if not os.path.exists("/content/Applio"):
    run("git clone -q --depth 1 https://github.com/IAHispano/Applio.git /content/Applio")
run("micromamba run -n rvc pip install -q -r requirements.txt", cwd="/content/Applio", tail=40)
print(run("micromamba run -n rvc python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\""))
# pretrained RVC v2 base models, RMVPE pitch model, ContentVec embedder
print(run("micromamba run -n rvc python core.py prerequisites --pretraineds_hifigan True --models True --exe False",
          cwd="/content/Applio", tail=40)[-1500:])

In [ ]:
#@title 5) Train the voice (preprocess → features → training → index)
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
EPOCHS = 150   #@param {type:"integer"}
need("/content/dataset/000.wav", 3)
t0 = time.time()
print(core("preprocess", "--model_name", NAME, "--dataset_path", "/content/dataset", "--sample_rate", 40000,
           "--cpu_cores", os.cpu_count())[-800:])
print(core("extract", "--model_name", NAME, "--f0_method", "rmvpe", "--sample_rate", 40000, "--gpu", 0,
           "--cpu_cores", os.cpu_count())[-800:])
print(core("train", "--model_name", NAME, "--total_epoch", EPOCHS, "--save_every_epoch", 25, "--sample_rate", 40000,
           "--batch_size", 8, "--gpu", 0, "--pretrained", True, "--save_only_latest", True, tail=60)[-1500:])
print(core("index", "--model_name", NAME)[-800:])
print(f"done in {(time.time() - t0) / 60:.0f} min")
print("\n".join(sorted(glob.glob(f"{A}/logs/{NAME}/*.pth") + glob.glob(f"{A}/logs/{NAME}/*.index"))))

In [ ]:
#@title 6) Test: a different voice → our new voice
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
CHOSEN = int(open("/content/chosen.txt").read()) if os.path.exists("/content/chosen.txt") else 0
REF = f"/content/cand/voice_{CHOSEN}.wav"
need("/content/test_other_voice.wav", 3)
if not glob.glob(f"{A}/logs/{NAME}/{NAME}*.pth"): raise RuntimeError("no trained model yet - run cell 5 first")
pth = sorted(glob.glob(f"{A}/logs/{NAME}/{NAME}*.pth"), key=os.path.getmtime)[-1]
idx = sorted(glob.glob(f"{A}/logs/{NAME}/*.index"), key=os.path.getmtime)[-1]
print(pth, idx)
print(core("infer", "--input_path", "/content/test_other_voice.wav", "--output_path", "/content/test_converted.wav",
           "--pth_path", pth, "--index_path", idx, "--pitch", 0, "--f0_method", "rmvpe", "--index_rate", 0.75)[-600:])
print("before (other voice):"); ipd.display(ipd.Audio("/content/test_other_voice.wav"))
print("after (our voice):");    ipd.display(ipd.Audio("/content/test_converted.wav"))
print("reference (chosen voice):"); ipd.display(ipd.Audio(REF))

In [ ]:
#@title 7) Download the model (put the zip in voice_clone/work/rvc/ on the laptop)
# --- common setup (every cell has it, so a restart or a skipped cell does not break anything) ---
import os, glob, time, json, subprocess
import IPython.display as ipd
def run(cmd, cwd=None, tail=25):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        lines = [l for l in out.splitlines() if l.strip()]
        raise RuntimeError(f"FAILED: {cmd if isinstance(cmd, str) else ' '.join(cmd)}\n" + "\n".join(lines[-tail:]))
    return out
def need(path, cell):
    if not os.path.exists(path):
        raise RuntimeError(f"{path} is missing - run cell {cell} first")
def vox(jobs):
    need("/content/vox_gen.py", 2)
    json.dump(jobs, open("/content/vox_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    print(run("micromamba run -n vox python /content/vox_gen.py /content/vox_jobs.json", tail=40)[-1200:])
NAME, A, GREETING = "galaxy_voice", "/content/Applio", "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
def core(*args, tail=40):
    need(A, 4)
    return run(["micromamba", "run", "-n", "rvc", "python", "core.py", *map(str, args)], cwd=A, tail=tail)
# ---
CHOSEN = int(open("/content/chosen.txt").read()) if os.path.exists("/content/chosen.txt") else 0
REF = f"/content/cand/voice_{CHOSEN}.wav"
pth = sorted(glob.glob(f"{A}/logs/{NAME}/{NAME}*.pth"), key=os.path.getmtime)[-1:]
idx = sorted(glob.glob(f"{A}/logs/{NAME}/*.index"), key=os.path.getmtime)[-1:]
if not pth or not idx: raise RuntimeError("no trained model yet - run cell 5 first")
pth, idx = pth[0], idx[0]
import shutil
os.makedirs("/content/out", exist_ok=True)
for f in (pth, idx, REF):
    shutil.copy(f, "/content/out/")
open("/content/out/README.txt", "w").write("Fictional voice made with VoxCPM2 (seed %d) and RVC v2 (Applio). No real person's voice.\n" % CHOSEN)
shutil.make_archive("/content/galaxy_voice_rvc", "zip", "/content/out")
print(os.path.getsize("/content/galaxy_voice_rvc.zip") // 1_000_000, "MB")
try:
    from google.colab import files
    files.download("/content/galaxy_voice_rvc.zip")
except Exception as ex:
    print("download not available here:", type(ex).__name__, "- use the Colab file browser (/content/galaxy_voice_rvc.zip)")